In [15]:
from pathlib import Path

print("Notebook folder:", Path.cwd())
print("Data files:", [
    p.name for p in Path.cwd().iterdir()
    if p.suffix.lower() in {".csv",".zip", ".xlsx", ".xls"}
])

Notebook folder: c:\Users\jessi\Desktop\BT4211\final_project
Data files: ['spotify_dataset.csv']


In [16]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\jessi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [17]:
import pandas as pd

sheets = pd.read_csv("spotify_dataset.csv")

print("Sheets and sizes:", {name: sheet.shape for name, sheet in sheets.items()})


Sheets and sizes: {'Unnamed: 0': (114000,), 'track_id': (114000,), 'artists': (114000,), 'album_name': (114000,), 'track_name': (114000,), 'popularity': (114000,), 'duration_ms': (114000,), 'explicit': (114000,), 'danceability': (114000,), 'energy': (114000,), 'key': (114000,), 'loudness': (114000,), 'mode': (114000,), 'speechiness': (114000,), 'acousticness': (114000,), 'instrumentalness': (114000,), 'liveness': (114000,), 'valence': (114000,), 'tempo': (114000,), 'time_signature': (114000,), 'track_genre': (114000,)}


In [18]:
from pathlib import Path
import pandas as pd
from IPython.display import display

csv_path = Path.cwd() / "spotify_dataset.csv"
print(f"Notebook current folder: {Path.cwd()}")
print(f"CSV path: {csv_path}")
print(f"File exists: {csv_path.is_file()}")

raw_df = None
audit_df = None
if not csv_path.is_file():
    print("Could not load the dataset because spotify_dataset.csv is not in the notebook's current folder.")
    print("Fix: place the file in the folder shown above, or update csv_path to its actual location.")
else:
    try:
        # Use pandas defaults; do not guess an encoding or delimiter.
        raw_df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f"CSV load failed: {type(exc).__name__}: {exc}")
        print("Fix: verify the file is a valid CSV and confirm its encoding and delimiter from its source; then set the appropriate read_csv options explicitly.")

if raw_df is not None:
    print(f"\nRows: {raw_df.shape[0]:,}")
    print(f"Columns: {raw_df.shape[1]:,}")
    print("\nColumn names and data types:")
    display(pd.DataFrame({"column": raw_df.columns, "dtype": raw_df.dtypes.astype(str).values}))
    print("\nFirst five rows:")
    display(raw_df.head())

    row_count = len(raw_df)
    index_like = [
        col for col in raw_df.columns
        if str(col).strip().lower() == "index"
        or str(col).strip().lower().startswith("unnamed:")
    ]
    sequential_columns = []
    for col in raw_df.columns:
        numeric_values = pd.to_numeric(raw_df[col], errors="coerce")
        sequential_zero = numeric_values.notna().all() and numeric_values.reset_index(drop=True).tolist() == list(range(row_count))
        sequential_one = numeric_values.notna().all() and numeric_values.reset_index(drop=True).tolist() == list(range(1, row_count + 1))
        if sequential_zero or sequential_one:
            sequential_columns.append(col)

    print("\nExported-index review:")
    if index_like:
        for col in index_like:
            print(f"- {col!r}: appears index-like by name; sequential 0..{row_count - 1} or 1..{row_count}: {col in sequential_columns}")
    else:
        print("- No columns named like an exported index (e.g. 'Unnamed: 0' or 'index').")
    other_sequential = [col for col in sequential_columns if col not in index_like]
    if other_sequential:
        print(f"- Other columns exactly sequential from 0 or 1: {other_sequential}")
    if not index_like and not other_sequential:
        print("- No sequential index-like columns identified.")
    print("No columns have been dropped or altered.")

Notebook current folder: c:\Users\jessi\Desktop\BT4211\final_project
CSV path: c:\Users\jessi\Desktop\BT4211\final_project\spotify_dataset.csv
File exists: True

Rows: 114,000
Columns: 21

Column names and data types:


,column,dtype
0,Unnamed: 0,int64
1,track_id,object
2,artists,object
3,album_name,object
4,track_name,object
5,popularity,int64
6,duration_ms,int64
7,explicit,bool
8,danceability,float64
9,energy,float64



First five rows:


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic



Exported-index review:
- 'Unnamed: 0': appears index-like by name; sequential 0..113999 or 1..114000: True
No columns have been dropped or altered.


In [21]:
if raw_df is None:
    print("Audit checks were not run because the CSV did not load. See the load error or missing-file message above.")
else:
    # Normalize blank text only on a deep copy; raw_df remains unchanged.
    audit_df = raw_df.copy(deep=True)
    audit_df = audit_df.replace(r"^\s*$", pd.NA, regex=True)
    print(f"audit_df created as a separate copy: {audit_df.shape}; raw_df is unchanged.")

    missing_table = pd.DataFrame({
        "missing_count": audit_df.isna().sum(),
        "missing_percent": (audit_df.isna().mean() * 100).round(2),
    })
    print("\nMissing values by column (blank/whitespace-only text treated as missing in audit_df):")
    display(missing_table)
    rows_with_missing = int(audit_df.isna().any(axis=1).sum())
    print(f"Rows with at least one missing value: {rows_with_missing:,} of {len(audit_df):,}")

    duplicate_after_first = audit_df.duplicated()
    duplicate_group_members = audit_df.duplicated(keep=False)
    print("\nExact duplicate rows across all columns:")
    print(f"Rows duplicated after the first copy (duplicated()): {int(duplicate_after_first.sum()):,}")
    print(f"All rows belonging to duplicate groups (duplicated(keep=False)): {int(duplicate_group_members.sum()):,}")
    if duplicate_group_members.any():
        print("All rows belonging to exact-duplicate groups:")
        display(audit_df.loc[duplicate_group_members])
    else:
        print("No exact duplicate groups found.")

    repeated_id_count = 0
    repeated_id_rows = 0
    multi_genre_ids = 0
    conflict_counts = {}
    conflict_examples = []
    audio_feature_cols = [
        "danceability", "energy", "speechiness", "acousticness",
        "instrumentalness", "liveness", "valence",
    ]
    conflict_value_cols = [
        col for col in ["popularity", *audio_feature_cols]
        if col in audit_df.columns
    ]
    if "track_id" in audit_df.columns:
        id_counts = audit_df["track_id"].value_counts(dropna=True)
        repeated_ids = id_counts[id_counts > 1]
        repeated_id_count = len(repeated_ids)
        repeated_id_rows = int(audit_df["track_id"].isin(repeated_ids.index).sum())
        print("\nRepeated track_id values:")
        print(f"Distinct track_id values repeated: {repeated_id_count:,}")
        print(f"Rows carrying those repeated IDs: {repeated_id_rows:,}")

        for col in conflict_value_cols:
            conflict_counts[col] = 0
        for track_id, group in audit_df.loc[audit_df["track_id"].isin(repeated_ids.index)].groupby("track_id", sort=False):
            genres = []
            if "track_genre" in group.columns:
                genres = group["track_genre"].dropna().unique().tolist()
                if len(genres) > 1:
                    multi_genre_ids += 1
            conflicting_cols = []
            for col in conflict_value_cols:
                if group[col].nunique(dropna=True) > 1:
                    conflict_counts[col] += 1
                    conflicting_cols.append(col)
            if len(genres) > 1 or conflicting_cols:
                conflict_examples.append({
                    "track_id": track_id,
                    "rows": len(group),
                    "genres": genres,
                    "conflicting_popularity/audio_features": conflicting_cols,
                })
        if "track_genre" in audit_df.columns:
            print(f"Repeated IDs with multiple non-missing track_genre values: {multi_genre_ids:,}")
        else:
            print("Genre comparison not completed: track_genre column is absent.")
        print("Repeated IDs with conflicting non-missing popularity/audio-feature values by column:")
        display(pd.Series(conflict_counts, name="repeated_IDs_with_conflicts", dtype="int64").to_frame())
        print("Up to 10 repeated-ID examples with multiple genres and/or conflicting values:")
        display(pd.DataFrame(conflict_examples[:10]))
        if not repeated_id_count:
            print("No repeated non-missing track_id values found.")
    else:
        print("\nRepeated track_id check not completed: track_id column is absent.")

    print("\nBasic structure:")
    if "track_genre" in audit_df.columns:
        genre_count = int(audit_df["track_genre"].nunique(dropna=True))
        print(f"Unique non-missing genres: {genre_count:,}")
        print("Rows per genre (including missing genre labels if present):")
        display(audit_df["track_genre"].value_counts(dropna=False).rename_axis("track_genre").to_frame("rows"))
    else:
        genre_count = None
        print("Genre summary not completed: track_genre column is absent.")
    if "artists" in audit_df.columns:
        artist_count = int(audit_df["artists"].nunique(dropna=True))
        print(f"Unique non-missing artists: {artist_count:,}")
    else:
        artist_count = None
        print("Artist count not completed: artists column is absent.")
    row_count = len(audit_df)
    approximate_114k = abs(row_count - 114_000) <= 1_140
    print(f"Actual row count: {row_count:,}; approximately 114,000 (within 1%): {approximate_114k} (difference: {row_count - 114_000:+,}).")

    def numeric_review(column, lower=None, upper=None, allowed_values=None, positive=False):
        series = audit_df[column]
        numeric = pd.to_numeric(series, errors="coerce")
        nonnumeric = series.notna() & numeric.isna()
        valid = numeric.dropna()
        result = {
            "column": column,
            "dtype": str(series.dtype),
            "minimum": valid.min() if not valid.empty else pd.NA,
            "maximum": valid.max() if not valid.empty else pd.NA,
            "non_numeric_count": int(nonnumeric.sum()),
            "non_numeric_examples": series[nonnumeric].astype(str).unique().tolist()[:10],
        }
        if lower is not None or upper is not None:
            out_of_range = pd.Series(False, index=series.index)
            if lower is not None:
                out_of_range |= numeric < lower
            if upper is not None:
                out_of_range |= numeric > upper
            result["out_of_range_count"] = int(out_of_range.sum())
            result["out_of_range_examples"] = numeric[out_of_range].dropna().unique().tolist()[:10]
        if allowed_values is not None:
            unusual = numeric.notna() & ~numeric.isin(allowed_values)
            result["unusual_count"] = int(unusual.sum())
            result["unusual_examples"] = numeric[unusual].unique().tolist()[:10]
        if positive:
            unusual = numeric.notna() & (numeric <= 0)
            result["nonpositive_count"] = int(unusual.sum())
            result["nonpositive_examples"] = numeric[unusual].unique().tolist()[:10]
        return result

    print("\nPlausible-value checks (values are reported for review; nothing is changed):")
    required_value_columns = [
        "popularity", "duration_ms", *audio_feature_cols,
        "key", "mode", "time_signature", "tempo", "loudness", "explicit",
    ]
    absent_columns = [col for col in required_value_columns if col not in audit_df.columns]
    if absent_columns:
        print(f"Checks not completed because columns are absent: {absent_columns}")

    if "popularity" in audit_df.columns:
        popularity_review = numeric_review("popularity", lower=0, upper=100)
        print(f"popularity: dtype={popularity_review['dtype']}; outside 0-100 count={popularity_review['out_of_range_count']}; examples={popularity_review['out_of_range_examples']}; non-numeric count={popularity_review['non_numeric_count']}; examples={popularity_review['non_numeric_examples']}")
    if "duration_ms" in audit_df.columns:
        duration_review = numeric_review("duration_ms")
        duration_numeric = pd.to_numeric(audit_df["duration_ms"], errors="coerce")
        invalid_duration = duration_numeric.notna() & (duration_numeric <= 0)
        print(f"duration_ms: dtype={duration_review['dtype']}; zero/negative count={int(invalid_duration.sum())}; examples={duration_numeric[invalid_duration].unique().tolist()[:10]}; non-numeric count={duration_review['non_numeric_count']}; examples={duration_review['non_numeric_examples']}")
    for col in audio_feature_cols:
        if col in audit_df.columns:
            feature_review = numeric_review(col, lower=0, upper=1)
            print(f"{col}: outside 0-1 count={feature_review['out_of_range_count']}; examples={feature_review['out_of_range_examples']}; non-numeric count={feature_review['non_numeric_count']}; examples={feature_review['non_numeric_examples']}")

    domain_notes = {
        "key": "review values outside the common 0-11 range",
        "mode": "review values outside {0, 1}",
        "time_signature": "review values outside the common 3-7 range",
        "tempo": "review non-positive values",
        "loudness": "review the reported extremes; no cutoff is imposed",
    }
    for col in ["key", "mode", "time_signature", "tempo", "loudness"]:
        if col not in audit_df.columns:
            continue
        if col == "mode":
            review = numeric_review(col, allowed_values=[0, 1])
        elif col == "key":
            review = numeric_review(col, lower=0, upper=11)
        elif col == "time_signature":
            review = numeric_review(col, lower=3, upper=7)
        elif col == "tempo":
            review = numeric_review(col, positive=True)
        else:
            review = numeric_review(col)
        print(f"{col}: min={review['minimum']}; max={review['maximum']}; non-numeric count={review['non_numeric_count']}; examples={review['non_numeric_examples']}; {domain_notes[col]}.")
        if "out_of_range_count" in review:
            print(f"  Outside common range count={review['out_of_range_count']}; examples={review['out_of_range_examples']}")
        if "unusual_count" in review:
            print(f"  Not in {0, 1} count={review['unusual_count']}; examples={review['unusual_examples']}")
        if "nonpositive_count" in review:
            print(f"  Non-positive count={review['nonpositive_count']}; examples={review['nonpositive_examples']}")

    if "explicit" in audit_df.columns:
        explicit = audit_df["explicit"]
        explicit_values = explicit.dropna().unique().tolist()
        print(f"explicit: dtype={explicit.dtype}; unique non-missing values={explicit_values}; missing count={int(explicit.isna().sum())}")

    absent_structure = [col for col in ["track_id", "track_genre", "artists"] if col not in audit_df.columns]
    print("\nConcise audit summary:")
    print(f"- {row_count:,} rows x {audit_df.shape[1]:,} columns; approximately 114,000 rows within 1%: {approximate_114k}.")
    print(f"- {rows_with_missing:,} rows have at least one missing value after blank-text checks in audit_df.")
    print(f"- Exact duplicate rows after first copy: {int(duplicate_after_first.sum()):,}; all duplicate-group rows: {int(duplicate_group_members.sum()):,}.")
    if "track_id" in audit_df.columns:
        print(f"- Repeated track IDs: {repeated_id_count:,} distinct IDs across {repeated_id_rows:,} rows; no IDs were removed.")
    if "track_genre" in audit_df.columns:
        print(f"- Unique genres: {genre_count:,}.")
    if "artists" in audit_df.columns:
        print(f"- Unique artists: {artist_count:,}.")
    if absent_columns or absent_structure:
        print(f"- Checks not completed due to absent columns: {sorted(set(absent_columns + absent_structure))}.")
    else:
        print("- All requested check columns were present.")
    print("- This was an audit only: raw_df was not modified; no rows were dropped and no values were imputed or cleaned.")

audit_df created as a separate copy: (114000, 21); raw_df is unchanged.

Missing values by column (blank/whitespace-only text treated as missing in audit_df):


,missing_count,missing_percent
Unnamed: 0,0,0.0
track_id,0,0.0
artists,1,0.0
album_name,1,0.0
track_name,1,0.0
popularity,0,0.0
duration_ms,0,0.0
explicit,0,0.0
danceability,0,0.0
energy,0,0.0


Rows with at least one missing value: 1 of 114,000

Exact duplicate rows across all columns:
Rows duplicated after the first copy (duplicated()): 0
All rows belonging to duplicate groups (duplicated(keep=False)): 0
No exact duplicate groups found.

Repeated track_id values:
Distinct track_id values repeated: 16,641
Rows carrying those repeated IDs: 40,900
Repeated IDs with multiple non-missing track_genre values: 16,299
Repeated IDs with conflicting non-missing popularity/audio-feature values by column:


,repeated_IDs_with_conflicts
popularity,720
danceability,0
energy,0
speechiness,0
acousticness,0
instrumentalness,0
liveness,0
valence,0


Up to 10 repeated-ID examples with multiple genres and/or conflicting values:


,track_id,rows,genres,conflicting_popularity/audio_features
0,5SuOikwiRyPMVoIQDJUgSV,4,"[acoustic, j-pop, singer-songwriter, songwriter]",[]
1,4qPNDBW1i3p13qLCt0Ki3A,2,"[acoustic, chill]",[]
2,01MVOl9KtVTNfFiBU9I7dc,2,"[acoustic, indie-pop]",[]
3,6Vc5wAMmXdKIAM7WUoEb7N,2,"[acoustic, piano]",[]
4,1EzrEOXmMH3G43AXT1y7pA,2,"[acoustic, rock]",[]
5,7k9GuJYLp2AzqokyEdwEw2,2,"[acoustic, piano]",[]
6,4ptDJbJl35d7gQfeNteBwp,2,"[acoustic, chill]",[]
7,0X9MxHR1rTkEHDjp95F2OO,2,"[acoustic, chill]",[]
8,6xKeQgzfjixSUld14qUezm,2,"[acoustic, guitar]",[]
9,4Yo0igmcoNyat1secaH0OD,2,"[acoustic, guitar]",[]



Basic structure:
Unique non-missing genres: 114
Rows per genre (including missing genre labels if present):


,rows
track_genre,
acoustic,1000
punk-rock,1000
progressive-house,1000
power-pop,1000
pop,1000
...,...
folk,1000
emo,1000
electronic,1000


Unique non-missing artists: 31,437
Actual row count: 114,000; approximately 114,000 (within 1%): True (difference: +0).

Plausible-value checks (values are reported for review; nothing is changed):
popularity: dtype=int64; outside 0-100 count=0; examples=[]; non-numeric count=0; examples=[]
duration_ms: dtype=int64; zero/negative count=1; examples=[0]; non-numeric count=0; examples=[]
danceability: outside 0-1 count=0; examples=[]; non-numeric count=0; examples=[]
energy: outside 0-1 count=0; examples=[]; non-numeric count=0; examples=[]
speechiness: outside 0-1 count=0; examples=[]; non-numeric count=0; examples=[]
acousticness: outside 0-1 count=0; examples=[]; non-numeric count=0; examples=[]
instrumentalness: outside 0-1 count=0; examples=[]; non-numeric count=0; examples=[]
liveness: outside 0-1 count=0; examples=[]; non-numeric count=0; examples=[]
valence: outside 0-1 count=0; examples=[]; non-numeric count=0; examples=[]
key: min=0; max=11; non-numeric count=0; examples=[]; rev